# Eksperyment: Cechy zmęczenia / Fatigue (Sprint 3b)

## Cel
Czy uwzględnienie zmęczenia gracza poprawia predykcje? Dwie nowe cechy (liczone bez leakage,
z chronologicznego indeksu): **rest_days** (dni od ostatniego meczu, cap 60) oraz
**tourney_minutes** (minuty zagrane w bieżącym turnieju -- skumulowane przez wcześniejsze rundy).
Symetryzowane do p1/p2 + różnice. Te same tuned HP co baseline (ablation).

In [1]:
import sys, io, contextlib, runpy
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
sys.path.insert(0, str(Path("../src").resolve()))
from tennis_model_fatigue import compute_fatigue_for_2024, NEW_FEATURES, data_file, TARGET_YEAR, HISTORY_START_YEAR
print("Rok docelowy:", TARGET_YEAR, "| nowe cechy:", NEW_FEATURES)

Rok docelowy: 2025 | nowe cechy: ['p1_rest_days', 'p2_rest_days', 'rest_days_diff', 'p1_tourney_minutes', 'p2_tourney_minutes', 'tourney_minutes_diff']


## 1. Reuse baseline pipeline

In [2]:
BASE = Path("../src/tennis_model.py").resolve()
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    ns = runpy.run_path(str(BASE))
symmetrize_data = ns["symmetrize_data"]
compute_symmetric_match_evaluation = ns["compute_symmetric_match_evaluation"]
evaluate_calibration_quality = ns["evaluate_calibration_quality"]
baseline_search = ns["search"]
RANDOM_STATE = ns["RANDOM_STATE"]
base_features = list(ns["features"]); cols_base = list(ns["cols_base"])
df_train_raw = ns["df_train_raw"].copy(); df_val_raw = ns["df_val_raw"].copy(); df_test_raw = ns["df_test_raw"].copy()
baseline_val_acc = float(ns["val_acc"]); baseline_test_acc = float(ns["test_acc"]); baseline_match_acc = float(ns["match_accuracy"])
print(f"Baseline: val={baseline_val_acc:.4f}  test={baseline_test_acc:.4f}  match={baseline_match_acc:.4f}  (cech: {len(base_features)})")

Baseline: val=0.6106  test=0.6604  match=0.6566  (cech: 40)


## 2. Liczenie cech zmęczenia (leakage-safe)
`compute_fatigue_for_2024` przetwarza historię + rok docelowy chronologicznie i dla każdego meczu
liczy rest_days oraz tourney_minutes ze ŚCIŚLE wcześniejszych meczów (indeks + bisect).

In [3]:
full_target = pd.read_csv(data_file(TARGET_YEAR))
full_target["tourney_date"] = pd.to_datetime(full_target["tourney_date"], format="%Y%m%d")
full_target = full_target.sort_values(["tourney_date", "match_num"]).reset_index(drop=True)
full_target_base = full_target[cols_base + ["tourney_id", "minutes"]].dropna(subset=cols_base).reset_index(drop=True)
n_train, n_val, n_test = len(df_train_raw), len(df_val_raw), len(df_test_raw)
assert len(full_target_base) == n_train + n_val + n_test
fatigue = compute_fatigue_for_2024(full_target_base)
fat_train = fatigue.iloc[:n_train].reset_index(drop=True)
fat_val = fatigue.iloc[n_train:n_train + n_val].reset_index(drop=True)
fat_test = fatigue.iloc[n_train + n_val:].reset_index(drop=True)
print(fatigue.describe().round(1).to_string())

       w_rest_days  l_rest_days  w_tourney_minutes  l_tourney_minutes
count       2647.0       2647.0             2647.0             2647.0
mean          10.4         11.0               86.8               98.0
std           15.6         17.4              129.6              137.5
min            0.0          0.0                0.0                0.0
25%            0.0          0.0                0.0                0.0
50%            6.0          0.0                0.0                0.0
75%           14.0         14.0              139.0              156.0
max           60.0         60.0              904.0              964.0


## 3. Doklejenie + symetryzacja (w_/l_ -> p1_/p2_) + różnice

In [4]:
def attach(df_raw, fat):
    df_raw = df_raw.copy().reset_index(drop=True)
    for col in ("w_rest_days", "l_rest_days", "w_tourney_minutes", "l_tourney_minutes"):
        df_raw[col] = fat[col].to_numpy()
    return df_raw
df_train_raw = attach(df_train_raw, fat_train); df_val_raw = attach(df_val_raw, fat_val); df_test_raw = attach(df_test_raw, fat_test)

def build_split(df_raw, shuffle):
    sym = symmetrize_data(df_raw, shuffle=shuffle)
    raw_map = df_raw[["match_id", "w_rest_days", "l_rest_days", "w_tourney_minutes", "l_tourney_minutes"]]
    sym = sym.merge(raw_map, on="match_id", how="left", validate="many_to_one")
    w = (sym["y"] == 1).to_numpy()
    sym["p1_rest_days"] = np.where(w, sym["w_rest_days"], sym["l_rest_days"])
    sym["p2_rest_days"] = np.where(w, sym["l_rest_days"], sym["w_rest_days"])
    sym["p1_tourney_minutes"] = np.where(w, sym["w_tourney_minutes"], sym["l_tourney_minutes"])
    sym["p2_tourney_minutes"] = np.where(w, sym["l_tourney_minutes"], sym["w_tourney_minutes"])
    sym["rest_days_diff"] = sym["p1_rest_days"] - sym["p2_rest_days"]
    sym["tourney_minutes_diff"] = sym["p1_tourney_minutes"] - sym["p2_tourney_minutes"]
    return sym
train_data = build_split(df_train_raw, True); val_data = build_split(df_val_raw, True); test_data = build_split(df_test_raw, True)
features = base_features + NEW_FEATURES
print("Cech razem:", len(features))

Cech razem: 46


## 4. Trening RF + ewaluacja

In [5]:
X_train, y_train = train_data[features], train_data["y"]
X_val, y_val = val_data[features], val_data["y"]; X_test, y_test = test_data[features], test_data["y"]
best_rf = RandomForestClassifier(**baseline_search.best_params_, n_jobs=-1, random_state=RANDOM_STATE)
best_rf.fit(X_train, y_train)
val_acc = float(accuracy_score(y_val, best_rf.predict(X_val))); test_acc = float(accuracy_score(y_test, best_rf.predict(X_test)))
proba = best_rf.predict_proba(X_test)[:, 1]; test_data["p1_win_probability"] = proba
_, match_acc = compute_symmetric_match_evaluation(test_data); q = evaluate_calibration_quality(y_test.to_numpy(), proba)
imp = pd.DataFrame({"feature": features, "importance": best_rf.feature_importances_}).sort_values("importance", ascending=False).reset_index(drop=True); imp["rank"] = imp.index + 1
print(f"baseline       match={baseline_match_acc:.4f}")
print(f"+fatigue       match={match_acc:.4f}  Brier={q['brier_score']:.4f}  DELTA={match_acc-baseline_match_acc:+.4f}")
for f in NEW_FEATURES:
    r = imp[imp.feature == f].iloc[0]; print(f"  {f:<22} rank {int(r['rank']):>2}/{len(features)}")

baseline       match=0.6566
+fatigue       match=0.6566  Brier=0.2170  DELTA=+0.0000
  p1_rest_days           rank 37/46
  p2_rest_days           rank 39/46
  rest_days_diff         rank 34/46
  p1_tourney_minutes     rank 36/46
  p2_tourney_minutes     rank 35/46
  tourney_minutes_diff   rank 28/46


## Wnioski
Cechy zmęczenia (dni odpoczynku, minuty na korcie) praktycznie nie ruszają trafności. Na walidacji przez 6 sezonów wychodzi +0,03 p.p. (McNemar p = 1,0). Model z nich korzysta, ale przewaga się nie utrzymuje — mieści się w szumie.